In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"  # Ensure Keras 3 uses JAX backend

import keras
import keras_hub
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Load CIFAR-10 dataset
dataset_name = "cifar10"
dataset, dataset_info = tfds.load(
    dataset_name,
    as_supervised=True,
    with_info=True
)
data_train, data_test = dataset["train"], dataset["test"]

BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)
NUM_CLASSES = dataset_info.features['label'].num_classes

def preprocess_inputs(image, label):
    image = keras.ops.image.resize(image, IMAGE_SIZE)
    image = keras.ops.cast(image, "float32") / 255.0  # Normalize
    label = keras.ops.one_hot(label, NUM_CLASSES)
    return image, label

# Preprocess dataset
data_train = data_train.map(preprocess_inputs).batch(BATCH_SIZE)
data_test = data_test.map(preprocess_inputs).batch(BATCH_SIZE)

# Load ResNetV2-50 model from Keras Hub
classifier = keras_hub.models.ImageClassifier.from_preset("resnet_v2_50_imagenet", num_classes=NUM_CLASSES)

# Compile the model
classifier.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Train the model
history = classifier.fit(data_train, validation_data=data_test, epochs=5)

# Plot accuracy
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()


100%|██████████| 840/840 [00:00<00:00, 260kB/s]


100%|██████████| 3.54k/3.54k [00:00<00:00, 2.74MB/s]


100%|██████████| 98.1M/98.1M [00:00<00:00, 142MB/s]


100%|██████████| 90.2M/90.2M [00:00<00:00, 122MB/s]


Epoch 1/5
1396/1563 ━━━━━━━━━━━━━━━━━━━━ 1:08:03 24s/step - accuracy: 0.1214 - loss: 8.9181